In [1]:
# 用于学习优化器的用法，以CIFAR 10为例，演示如何使用优化器来更新模型的参数

import torch
import torchvision as tv
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter


In [2]:
# 数据准备
dataset = tv.datasets.CIFAR10(root="./dataset", train=True, transform=tv.transforms.ToTensor(), download=True)
dataloader = DataLoader(dataset, batch_size=64, drop_last=True)

class TryConv(nn.Module):
    def __init__(self):
        super().__init__()
        self.sequential = nn.Sequential(
            nn.Conv2d(in_channels = 3, out_channels = 32, kernel_size = 5, stride = 1, padding = 2),
            nn.MaxPool2d(kernel_size = 2),
            nn.Conv2d(in_channels = 32, out_channels = 32, kernel_size = 5, stride = 1, padding = 2),
            nn.MaxPool2d(kernel_size = 2),
            nn.Conv2d(in_channels = 32, out_channels = 64, kernel_size = 5, stride = 1, padding = 2),
            nn.MaxPool2d(kernel_size = 2),
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 10)
        )

    def forward(self, x):
        x = self.sequential(x)
        return x

model_test = TryConv()
input = torch.ones((64, 3, 32, 32))
output = model_test(input)
writer = SummaryWriter("logs/sequential")
writer.add_graph(model_test, input)
writer.close()

# 定义损失函数和优化器
loss_fn = nn.CrossEntropyLoss()

# 接下来展示优化器的用法
optim = torch.optim.SGD(model_test.parameters(), lr = 0.01)

# 训练循环
num_epochs = 10

for epoch in range(num_epochs):
    running_loss = 0.0
    for data in dataloader:
        img, target = data
        op = model_test(img)
        loss = loss_fn(op, target)
        optim.zero_grad() # 这里需要清空梯度，否则会累积
        loss.backward() # 这里会计算梯度，存储在每个参数的.grad属性中
        optim.step() # 这里才会更新参数
        running_loss += loss.item()
    
    avg_loss = running_loss / len(dataloader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss}")
    


/opt/miniconda3/envs/unipt/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Epoch [1/10], Loss: 2.0453566251735222
Epoch [2/10], Loss: 1.7316887116157444
Epoch [3/10], Loss: 1.5817846006376337
Epoch [4/10], Loss: 1.4791499383928834
Epoch [5/10], Loss: 1.3968542817307494
Epoch [6/10], Loss: 1.3244178336347416
Epoch [7/10], Loss: 1.2603973395235255
Epoch [8/10], Loss: 1.2041006082151366
Epoch [9/10], Loss: 1.154778569898593
Epoch [10/10], Loss: 1.111378085140077
